In [ ]:
# ── CÉLULA DE SETUP — rode esta PRIMEIRA ─────────────────────────────────
import os
os.makedirs("reports/figures", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Faz upload do nps_processed.csv gerado pelo notebook 02
from google.colab import files
print("Selecione o arquivo nps_processed.csv")
uploaded = files.upload()
for nome, conteudo in uploaded.items():
    with open(nome, "wb") as f:
        f.write(conteudo)
print("Arquivo carregado!")

def salvar_figura(fig, nome):
    path = f"reports/figures/{nome}"
    fig.savefig(path, bbox_inches="tight", dpi=120)
    print(f"Figura salva: {path}")


# 03 · Modelo Preditivo de NPS

Este tópico tem como objetivo construir e comparar modelos capazes de prever o NPS antes da aplicação da pesquisa, utilizando os dados já processados pelo notebook 02, por meio das abordagens de **Regressão** e **Classificação**.

## Estratégia

### Abordagem A: Regressão
Estima a nota de NPS em escala contínua (0 a 10).
**Modelos:** Regressão Linear · Random Forest Regressor · Gradient Boosting Regressor

### Abordagem B: Classificação
Categoriza o cliente em Detrator / Neutro / Promotor.
**Modelos:** Decision Tree · Random Forest · Gradient Boosting · Logistic Regression · SVM

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score, accuracy_score
)

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120})
print("Bibliotecas carregadas!")


## 1. Carregamento dos Dados Processados


In [ ]:
df = pd.read_csv("nps_processed.csv")

# Separando features e targets
X = df.drop(columns=["nps_score", "nps_category"], errors="ignore")

print(f"Dataset carregado: {df.shape}")
print(f"Features disponíveis: {X.shape[1]}")
print(X.columns.tolist())


---
# PARTE A: REGRESSÃO
**Target:** `nps_score` (0 a 10)

**Métricas:** MAE · RMSE · R²


In [ ]:
y_reg = df["nps_score"].values

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.20, random_state=42
)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled  = scaler_r.transform(X_test_r)

print(f"Treino: {X_train_r.shape[0]:,} | Teste: {X_test_r.shape[0]:,}")


In [ ]:
modelos_regressao = {
    "Regressão Linear":          LinearRegression(),
    "Random Forest Regressor":   RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
}

resultados_regressao = []

for nome, modelo in modelos_regressao.items():
    if nome == "Regressão Linear":
        modelo.fit(X_train_r_scaled, y_train_r)
        y_pred = modelo.predict(X_test_r_scaled)
    else:
        modelo.fit(X_train_r, y_train_r)
        y_pred = modelo.predict(X_test_r)

    mae  = mean_absolute_error(y_test_r, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred))
    r2   = r2_score(y_test_r, y_pred)

    resultados_regressao.append({"Modelo": nome, "MAE": round(mae,3), "RMSE": round(rmse,3), "R²": round(r2,3)})
    print(f"{nome:30s} | MAE: {mae:.3f} | RMSE: {rmse:.3f} | R²: {r2:.3f}")

df_reg = pd.DataFrame(resultados_regressao).sort_values("R²", ascending=False)
print("\n--- Ranking por R² ---")
print(df_reg.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
metricas = ["MAE", "RMSE", "R²"]
cores    = ["#E74C3C", "#F39C12", "#2ECC71"]
for ax, metrica, cor in zip(axes, metricas, cores):
    d = df_reg.sort_values(metrica, ascending=(metrica != "R²"))
    ax.barh(d["Modelo"], d[metrica], color=cor, alpha=0.85)
    ax.set_title(metrica)
    ax.invert_yaxis()
fig.suptitle("Comparação dos Modelos de Regressão", fontsize=14, fontweight="bold")
plt.tight_layout()
salvar_figura(fig, "modelo_01_comparacao_regressao.png")
plt.show()


### Interpretação (Regressão)
- **MAE**: em média, quantos pontos o modelo erra na nota (escala 0-10)
- **R²**: quanto da variação do NPS os dados operacionais conseguem explicar
- Modelos de árvore capturam melhor relações não-lineares, como o ponto de ruptura no atraso de entrega


---
# PARTE B: CLASSIFICAÇÃO
**Target:** `nps_category` (Detrator / Neutro / Promotor)

**Métricas:** Acurácia · F1-Score Macro · Matriz de Confusão


In [ ]:
le = LabelEncoder()
y_class = le.fit_transform(df["nps_category"])
class_names = le.classes_

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.20, random_state=42, stratify=y_class
)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled  = scaler_c.transform(X_test_c)

print("Classes:", class_names)
print(f"Treino: {X_train_c.shape[0]:,} | Teste: {X_test_c.shape[0]:,}")
print("\nDistribuição no treino:")
print(pd.Series(le.inverse_transform(y_train_c)).value_counts())


In [ ]:
modelos_classificacao = {
    "Decision Tree":      DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=42),
    "Random Forest":      RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting":  GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "SVM":                SVC(kernel="rbf", class_weight="balanced", random_state=42),
}

usa_scaling = {"Logistic Regression", "SVM"}
resultados_class = []
predicoes = {}

for nome, modelo in modelos_classificacao.items():
    Xtr = X_train_c_scaled if nome in usa_scaling else X_train_c
    Xte = X_test_c_scaled  if nome in usa_scaling else X_test_c
    modelo.fit(Xtr, y_train_c)
    y_pred = modelo.predict(Xte)

    acc      = accuracy_score(y_test_c, y_pred)
    f1_macro = f1_score(y_test_c, y_pred, average="macro")
    f1_w     = f1_score(y_test_c, y_pred, average="weighted")

    resultados_class.append({"Modelo": nome, "Acurácia": round(acc,3), "F1-Macro": round(f1_macro,3), "F1-Weighted": round(f1_w,3)})
    predicoes[nome] = y_pred
    print(f"{nome:22s} | Acurácia: {acc:.3f} | F1-Macro: {f1_macro:.3f} | F1-Weighted: {f1_w:.3f}")

df_class = pd.DataFrame(resultados_class).sort_values("F1-Macro", ascending=False)
print("\n--- Ranking por F1-Macro ---")
print(df_class.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
d = df_class.sort_values("F1-Macro")
bars = ax.barh(d["Modelo"], d["F1-Macro"], color="#3498DB", alpha=0.85)
ax.set_xlabel("F1-Score Macro")
ax.set_title("Comparação dos Modelos de Classificação — F1-Score Macro")
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_xlim(0, 1)
plt.tight_layout()
salvar_figura(fig, "modelo_02_comparacao_classificacao.png")
plt.show()


In [ ]:
# Matriz de confusão do melhor modelo
melhor_nome = df_class.iloc[0]["Modelo"]
melhor_pred = predicoes[melhor_nome]

print(f"Melhor modelo: {melhor_nome}\n")
print(classification_report(y_test_c, melhor_pred, target_names=class_names))

fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test_c, melhor_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Matriz de Confusão — {melhor_nome}")
plt.tight_layout()
salvar_figura(fig, f"modelo_03_confusion_matrix.png")
plt.show()


### Interpretação (Classificação)
- A **Acurácia** pode enganar em bases desbalanceadas, portanto, optamos por priorizar o **F1-Score Macro**
- A classe **Neutro** costuma ser a mais difícil de prever, fazendo fronteira entre Detrator e Promotor
- **SVM** e **Logistic Regression** precisam de padronização, aplicada antes do treino


---
## Comparação Final: Regressão vs Classificação

| Critério | Regressão | Classificação |
|---|---|---|
| Saída | Nota estimada (ex.: 4.7) | Categoria (Detrator/Neutro/Promotor) |
| Quando usar | Ranquear clientes por grau de risco | Acionar fluxos de ação por grupo |
| Métrica chave | MAE / R² | F1-Score Macro |

**Recomendação:** Usar Classificação para alertas operacionais e Regressão para ranquear prioridade dentro de um grupo específico.


## Salvando os Melhores Modelos


In [ ]:
import joblib

melhor_nome_reg  = df_reg.iloc[0]["Modelo"]
melhor_nome_class = df_class.iloc[0]["Modelo"]

joblib.dump(modelos_regressao[melhor_nome_reg],      "models/nps_regressor.pkl")
joblib.dump(modelos_classificacao[melhor_nome_class],"models/nps_classifier.pkl")
joblib.dump(le,                                       "models/label_encoder.pkl")
joblib.dump(scaler_c,                                "models/scaler.pkl")
joblib.dump(X.columns.tolist(),                      "models/feature_names.pkl")

print(f"Melhor regressor:     {melhor_nome_reg}  → models/nps_regressor.pkl")
print(f"Melhor classificador: {melhor_nome_class} → models/nps_classifier.pkl")


In [ ]:
# ── CÉLULA FINAL — baixa figuras e modelos ───────────────────────────────
from google.colab import files
import os

print("Baixando figuras...")
for fig in sorted(os.listdir("reports/figures")):
    if fig.endswith(".png"):
        files.download(f"reports/figures/{fig}")

print("\nBaixando modelos...")
for m in sorted(os.listdir("models")):
    if m.endswith(".pkl"):
        files.download(f"models/{m}")

print("\nPronto! Mova os arquivos para as pastas corretas no repositório.")
